In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from arch import arch_model

# 1. 读取原始数据
df = pd.read_excel('data/LOCAL_INPUT')
r_500 = df['r_500'].dropna()
r_1000 = df['r_1000'].dropna()

# 2. 拟合边缘模型并提取标准化残差 (以中证500为例)
# 注：此处演示使用常数均值的 GJR-GARCH-t，用于快速提取残差
model_500 = arch_model(r_500, mean='Constant', vol='GARCH', p=1, o=1, q=1, dist='studentst')
res_500 = model_500.fit(disp='off')

# 提取标准化残差 z_hat
z_500 = res_500.resid.dropna() / res_500.conditional_volatility.dropna()
# 提取 5.1 节算出的真实自由度 nu (约等于 5.62)
nu_500 = res_500.params['nu']

# 3. 执行【概率积分变换 PIT】转换为 [0, 1] 均匀分布的 U
# stats.t.cdf 是标准的学生t分布累积分布函数
u = stats.t.cdf(z_500, df=nu_500)

# ---------------- 同理处理中证1000 ----------------
model_1000 = arch_model(r_1000, mean='Constant', vol='GARCH', p=1, o=1, q=1, dist='studentst')
res_1000 = model_1000.fit(disp='off')
z_1000 = res_1000.resid.dropna() / res_1000.conditional_volatility.dropna()
nu_1000 = res_1000.params['nu'] # 约等于 4.85

v = stats.t.cdf(z_1000, df=nu_1000)

# 4. 生成 Copula 拟合所需的最终数据集
copula_input = pd.DataFrame({'u': u, 'v': v})
print("数据处理完毕！u 和 v 序列已成功转换为 [0,1] 均匀分布。")
print(copula_input.head())

# 可以将处理好的数据导出，供后续 Copula 建模使用
# copula_input.to_csv('copula_uv_data.csv', index=False)

In [ ]:
import numpy as np
import scipy.stats as stats
from scipy.optimize import minimize

# ==========================================
# 假设 u 和 v 已经是你上一步通过 PIT 转换好的序列
# n 为样本量 (例如 1935)
# ==========================================
n = len(u)

# ==========================================
# 1. Gaussian Copula 拟合
# ==========================================
def neg_log_lik_gaussian(rho):
    if abs(rho) >= 0.999: return 1e10 # 边界惩罚
    x = stats.norm.ppf(u)
    y = stats.norm.ppf(v)
    term1 = -0.5 * np.log(1 - rho**2)
    term2 = -(rho**2 * (x**2 + y**2) - 2 * rho * x * y) / (2 * (1 - rho**2))
    return -np.sum(term1 + term2)

res_gauss = minimize(neg_log_lik_gaussian, x0=[0.8], bounds=[(0.01, 0.999)])
rho_g = res_gauss.x[0]
ll_g = -res_gauss.fun
aic_g = 2 * 1 - 2 * ll_g
bic_g = np.log(n) * 1 - 2 * ll_g

# ==========================================
# 2. Clayton Copula 拟合 (下尾相关)
# ==========================================
def neg_log_lik_clayton(theta):
    if theta <= 0: return 1e10
    term1 = np.log(theta + 1)
    term2 = -(theta + 1) * (np.log(u) + np.log(v))
    term3 = -(2 + 1/theta) * np.log(u**-theta + v**-theta - 1)
    return -np.sum(term1 + term2 + term3)

res_clayton = minimize(neg_log_lik_clayton, x0=[2.0], bounds=[(0.1, 30.0)])
theta_c = res_clayton.x[0]
ll_c = -res_clayton.fun
aic_c = 2 * 1 - 2 * ll_c
bic_c = np.log(n) * 1 - 2 * ll_c

# 计算 Clayton 下尾相关系数 (Lower Tail Dependence)
lambda_L_clayton = 2 ** (-1 / theta_c)

# ==========================================
# 3. Gumbel Copula 拟合 (上尾相关)
# ==========================================
def neg_log_lik_gumbel(theta):
    if theta <= 1: return 1e10
    x = -np.log(u)
    y = -np.log(v)
    a = x**theta + y**theta
    # Gumbel Copula 精确对数概率密度
    log_c = -a**(1/theta) + (theta - 1)*np.log(x*y) + (1/theta - 2)*np.log(a) + np.log(a**(1/theta) + theta - 1) - np.log(u*v)
    return -np.sum(log_c)

res_gumbel = minimize(neg_log_lik_gumbel, x0=[3.0], bounds=[(1.001, 20.0)])
theta_gum = res_gumbel.x[0]
ll_gum = -res_gumbel.fun
aic_gum = 2 * 1 - 2 * ll_gum
bic_gum = np.log(n) * 1 - 2 * ll_gum

# ==========================================
# 4. Student-t Copula 拟合 (双侧尾部相关)
# ==========================================
def neg_log_lik_t(params):
    rho, nu = params
    if abs(rho) >= 0.999 or nu <= 2: return 1e10
    x = stats.t.ppf(u, df=nu)
    y = stats.t.ppf(v, df=nu)
    
    # 二元 t 分布的对数密度 减去 两个边缘 t 分布的对数密度
    cholesky_det = 1 - rho**2
    log_pdf_biv = stats.multivariate_t.logpdf(np.column_stack((x, y)), df=nu, shape=[[1, rho], [rho, 1]])
    log_pdf_marg1 = stats.t.logpdf(x, df=nu)
    log_pdf_marg2 = stats.t.logpdf(y, df=nu)
    
    return -np.sum(log_pdf_biv - log_pdf_marg1 - log_pdf_marg2)

# t-Copula 有两个参数 (rho 和 自由度 nu)，需要更高的计算资源
res_t = minimize(neg_log_lik_t, x0=[0.9, 5.0], bounds=[(0.5, 0.999), (2.1, 30.0)])
rho_t, nu_t = res_t.x
ll_t = -res_t.fun
aic_t = 2 * 2 - 2 * ll_t  # 两个参数，所以是 2*2
bic_t = np.log(n) * 2 - 2 * ll_t

# ==========================================
# 5. 打印最终用于表格的结果
# ==========================================
print("=== 表 5-2 Copula 参数与拟合优度 ===")
print(f"Gaussian : rho = {rho_g:.4f} | LLF = {ll_g:.2f} | AIC = {aic_g:.2f} | BIC = {bic_g:.2f}")
print(f"Student-t: rho = {rho_t:.4f}, nu = {nu_t:.4f} | LLF = {ll_t:.2f} | AIC = {aic_t:.2f} | BIC = {bic_t:.2f}")
print(f"Gumbel   : theta = {theta_gum:.4f} | LLF = {ll_gum:.2f} | AIC = {aic_gum:.2f} | BIC = {bic_gum:.2f}")
print(f"Clayton  : theta = {theta_c:.4f} | LLF = {ll_c:.2f} | AIC = {aic_c:.2f} | BIC = {bic_c:.2f}")
print(f"\n=> 极其关键的结论：Clayton Copula 的极限下尾暴跌传染概率 λ_L = {lambda_L_clayton:.4f}")